In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
from torch.nn.init import kaiming_normal_

## Basic Blocks

In [5]:
# === Activation Dictionary ===
activation_dict = {
    'relu': F.relu,
    'leaky_relu': F.leaky_relu,
    'tanh': F.tanh,
    'sigmoid': F.sigmoid,
    'gelu': F.gelu
}

# === CBAM Attention ===
class ChannelAttention(nn.Module):
    def __init__(self, in_planes, ratio=16):
        super(ChannelAttention, self).__init__()
        hidden_planes = max(1, in_planes // ratio)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)

        self.shared_mlp = nn.Sequential(
            nn.Conv2d(in_planes, hidden_planes, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(hidden_planes, in_planes, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.shared_mlp(self.avg_pool(x))
        max_out = self.shared_mlp(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)


class SpatialAttention(nn.Module):
    def __init__(self, kernel_size=7):
        super(SpatialAttention, self).__init__()
        padding = kernel_size // 2
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x_cat = torch.cat([avg_out, max_out], dim=1)
        return self.sigmoid(self.conv(x_cat))

class CBAM(nn.Module):
    def __init__(self, channels, reduction_ratio=16, kernel_size=7):
        super(CBAM, self).__init__()
        self.channel_attention = ChannelAttention(channels, ratio=reduction_ratio)
        self.spatial_attention = SpatialAttention(kernel_size)

    def forward(self, x):
        out = x * self.channel_attention(x)
        out = out * self.spatial_attention(out)
        return out


# === FiLM Layer ===
class FiLM(nn.Module):
    def __init__(self, in_channels, cond_dim):
        super(FiLM, self).__init__()
        self.gamma = nn.Linear(cond_dim, in_channels)
        self.beta = nn.Linear(cond_dim, in_channels)

    def forward(self, x, cond):
        gamma = self.gamma(cond).unsqueeze(2).unsqueeze(3)
        beta = self.beta(cond).unsqueeze(2).unsqueeze(3)
        return gamma * x + beta


# === Residual Block ===
class ResBlock(nn.Module):
    def __init__(self, channels, cond_dim, activation_fn=F.leaky_relu):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.film1 = FiLM(channels, cond_dim)

        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.film2 = FiLM(channels, cond_dim)

        self.activation_fn = activation_fn
        self._initialize_weights()

    def _initialize_weights(self):
        kaiming_normal_(self.conv1.weight, mode='fan_out', nonlinearity='relu')
        kaiming_normal_(self.conv2.weight, mode='fan_out', nonlinearity='relu')

    def forward(self, x, cond):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.film1(out, cond)
        out = self.activation_fn(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.film2(out, cond)

        out += residual
        out = self.activation_fn(out)
        return out

class GlobalContextBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.mlp = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, H, W = x.shape
        context = self.pool(x).view(B, C)
        weights = self.mlp(context).view(B, C, 1, 1)
        return x * weights + x


class GlobalFeatureFusion(nn.Module):
    def __init__(self, in_channels, hidden_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, in_channels)
        )

    def forward(self, x):
        B, C, H, W = x.size()
        pooled = F.adaptive_avg_pool2d(x, 1).view(B, C)
        context = self.mlp(pooled).view(B, C, 1, 1)
        return x + context


class ASPP_SE(nn.Module):
    def __init__(self, in_channels, out_channels, reduction=16):
        super().__init__()
        self.atrous_block1 = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.atrous_block6 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=6, dilation=6)
        self.atrous_block12 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=12, dilation=12)
        self.atrous_block18 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=18, dilation=18)

        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(out_channels * 4, out_channels * 4 // reduction, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels * 4 // reduction, out_channels * 4, 1),
            nn.Sigmoid()
        )

        self.output = nn.Conv2d(out_channels * 4, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.atrous_block1(x)
        x2 = self.atrous_block6(x)
        x3 = self.atrous_block12(x)
        x4 = self.atrous_block18(x)

        x_cat = torch.cat([x1, x2, x3, x4], dim=1)
        se_weight = self.se(x_cat)
        x_cat = x_cat * se_weight

        return self.output(x_cat)

## Coupling Layer

In [6]:
class Coupling(nn.Module):
    def __init__(self, input_shape=(4, 91, 91), num_classes=10, reg=0.01, activation_fn=F.gelu,
                 normalization_type='groupnorm', filters=(32, 64, 128), verbose=False):
        super().__init__()
        self.input_shape = input_shape
        self.reg = reg
        self.activation_fn = activation_fn
        self.normalization_type = normalization_type
        self.verbose = verbose

        C, H, W = input_shape
        filter_1, filter_2, filter_3 = filters

        self.label_embedding = nn.Sequential(
            nn.Linear(num_classes, 128),
            nn.GELU()
        )

        self.conv1 = weight_norm(nn.Conv2d(C, filter_1, kernel_size=5, padding=2))
        self.bn1 = self.get_normalization(filter_1)
        self.film1 = FiLM(filter_1, 128)
        self.cbam1 = CBAM(filter_1)
        self.global1 = GlobalFeatureFusion(filter_1)
        self.aspp_early = ASPP_SE(filter_1, filter_1)

        self.conv2 = weight_norm(nn.Conv2d(filter_1, filter_2, kernel_size=3, padding=1))
        self.bn2 = self.get_normalization(filter_2)
        self.film2 = FiLM(filter_2, 128)
        self.cbam2 = CBAM(filter_2)
        self.resblock1 = ResBlock(filter_2, 128, activation_fn)
        self.global2 = GlobalContextBlock(filter_2)
        self.aspp_mid = ASPP_SE(filter_2, filter_2)

        self.conv3 = weight_norm(nn.Conv2d(filter_2, filter_3, kernel_size=3, padding=1))
        self.bn3 = self.get_normalization(filter_3)
        self.film3 = FiLM(filter_3, 128)
        self.cbam3 = CBAM(filter_3)
        self.resblock2 = ResBlock(filter_3, 128, activation_fn)
        self.global3 = GlobalContextBlock(filter_3)
        self.aspp = ASPP_SE(filter_3, filter_3)

        self.conv_s = nn.Sequential(
            nn.Conv2d(filter_3, C, kernel_size=3, padding=1),
            nn.Tanh()
        )
        self.conv_t = nn.Conv2d(filter_3, C, kernel_size=1)

    def get_normalization(self, num_features):
        if self.normalization_type == 'groupnorm':
            num_groups = min(8, num_features)
            while num_features % num_groups != 0:
                num_groups -= 1
            return nn.GroupNorm(num_groups, num_features)
        elif self.normalization_type == 'batchnorm':
            return nn.BatchNorm2d(num_features)
        else:
            raise ValueError(f"Unknown normalization type: {self.normalization_type}")

    def forward(self, x, y):
        if self.verbose:
            print(f"[Coupling Forward] Filters: conv1={self.conv1.out_channels}, conv2={self.conv2.out_channels}, conv3={self.conv3.out_channels}")
        #if self.verbose:
        #    print(f"Input x: {x.shape}")
        #    print(f"Input y: {y.shape}")

        y_emb = self.activation_fn(self.label_embedding(y))
        #if self.verbose:
        #    print(f"Label embedding: {y_emb.shape}")

        x = self.conv1(x)
        x = self.bn1(x)
        x = self.film1(x, y_emb)
        x = self.activation_fn(x)
        x = self.cbam1(x)
        x = self.global1(x)
        x = self.aspp_early(x)

        x = self.conv2(x)
        x = self.bn2(x)
        x = self.film2(x, y_emb)
        x = self.activation_fn(x)
        x = self.cbam2(x)
        x = self.resblock1(x, y_emb)
        x = self.global2(x)
        x = self.aspp_mid(x)

        x = self.conv3(x)
        x = self.bn3(x)
        x = self.film3(x, y_emb)
        x = self.activation_fn(x)
        x = self.cbam3(x)
        x = self.resblock2(x, y_emb)
        x = self.global3(x)
        x = self.aspp(x)

        s = 0.5 * self.conv_s(x)
        t = self.conv_t(x)

        #if self.verbose:
        #    print(f"Output s: {s.shape}")
        #    print(f"Output t: {t.shape}")

        return s, t